# 2.0版本控制台

In [ ]:
from backend2.l1.pipeline import run_l1_pipeline
from pathlib import Path
import time

# 默认只构造 h5_full 的 10%（N 轴等间隔抽样：1000 -> 100）
# 手动改为 True 可切回 100% 全构造
H5_FULL_CONSTRUCTION = False

h5_reader_cfg = {
    "kind": "h5",
    "path": "datasets/2D_rdb_NA_NA.h5",
    "dataset": "data",
    "fill_value": 0.0,
    "sample_ratio": 0.1,
    "sample_mode": "interval",
    "full_construction": H5_FULL_CONSTRUCTION,
}

configs = [
    {
        "dataset_id": "h5_full",
        "reader": h5_reader_cfg,
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "log_progress": True,
        "norm_chunk_n": 1,
    },
    {
        "dataset_id": "nc_full",
        "reader": {
            "kind": "nc",
            "path": "datasets/cylinder2d.nc",
            "var_keys": ["u", "v"],
            "time_key": "tdim",
            "y_key": "ydim",
            "x_key": "xdim",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "log_progress": True,
        "norm_chunk_n": 1,
    },
    {
        "dataset_id": "sst_full",
        "reader": {
            "kind": "mat",
            "path": "datasets/sst_weekly.mat",
            "var": "sst",
            "lon_key": "lon",
            "lat_key": "lat",
            "time_key": "time",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "log_progress": True,
        "norm_chunk_n": 1,
    },
]

summaries = []
total = len(configs)
overall_t0 = time.perf_counter()

for i, cfg in enumerate(configs, start=1):
    print(f"\n[L1] ({i}/{total}) start: {cfg['dataset_id']}", flush=True)
    t0 = time.perf_counter()
    summary = run_l1_pipeline(cfg)
    dt = time.perf_counter() - t0

    l1_dir = Path(summary.artifacts_dir)
    frozen_files = {
        "array5d_norm": str(l1_dir / "array5d_norm.npy"),
        "train_split": str(l1_dir / "splits" / "train.npy"),
        "val_split": str(l1_dir / "splits" / "val.npy"),
        "test_split": str(l1_dir / "splits" / "test.npy"),
        "stats_train": str(l1_dir / "stats_train.json"),
        "manifest": str(l1_dir / "manifest.json"),
    }

    summaries.append(
        {
            "dataset_id": summary.dataset_id,
            "shape5d": list(summary.shape5d),
            "split_sizes": summary.split_sizes,
            "stats_method": summary.stats_method,
            "artifacts_dir": summary.artifacts_dir,
            "elapsed_sec": round(dt, 2),
            "frozen_files": frozen_files,
        }
    )
    print(f"[L1] ({i}/{total}) done: {summary.dataset_id} in {dt:.2f}s", flush=True)

print(f"\n[L1] all done in {time.perf_counter() - overall_t0:.2f}s", flush=True)
summaries

In [ ]:
from backend2.l2.train import run_l2_train
from backend2.l2.infer import run_l2_infer
from backend2.l2.utils import now_tag
from backend2.l2.artifact_io import ArtifactManager
from backend2.l2.data import load_l1_array_mmap, load_split_pairs, PairDataset
from torch.utils.data import DataLoader
import json

datasets = ["h5_full", "nc_full", "sst_full"]
exp_name = "baseline_unet"

all_summaries = []

for dataset_id in datasets:
    run_name = f"run_{dataset_id}_{now_tag()}"

    manager = ArtifactManager(
        artifacts_dir="artifacts",
        dataset_id=dataset_id,
        exp_name=exp_name,
        run_name=run_name,
    )

    # 1) 直接从 L1 冻结产物构建 L2 DataLoader（mmap + splits）
    array5d, manifest = load_l1_array_mmap(manager)
    train_pairs = load_split_pairs(manager, array5d, manifest, "train", target_offset=1)
    train_ds = PairDataset(array5d=array5d, pairs=train_pairs, target_offset=1)
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
    first_batch = next(iter(train_loader))

    # 2) 训练与推理（L2.5 特征冻结默认开启）
    train_cfg = {
        "dataset_id": dataset_id,
        "artifacts_dir": "artifacts",
        "exp_name": exp_name,
        "run_name": run_name,
        "device": "auto",
        "seed": 123,
        "target_offset": 1,
        "batch_size": 8,
        "num_workers": 0,
        "epochs": 20,
        "lr": 1e-3,
        "model": {
            "base_channels": 32,
            "convs_per_stage": 2,
        },
    }
    infer_cfg = {
        "dataset_id": dataset_id,
        "artifacts_dir": "artifacts",
        "exp_name": exp_name,
        "run_name": run_name,
        "device": "auto",
        "target_offset": 1,
        "batch_size": 8,
        "num_workers": 0,
        "ckpt_name": "model_best.pt",
        "freeze_features": True,
        "freeze_layers": [
            "enc.stage1.out",
            "enc.stage2.out",
            "enc.stage3.out",
        ],
        "freeze_mode": "test",
        "model": {
            "base_channels": 32,
            "convs_per_stage": 2,
        },
        "probe": {
            "enabled": True,
            "record_level": 0,
            "hook_layers": ["enc.stage*.out", "dec.stage*.out", "skip.*", "head.out"],
        },
    }

    train_summary = run_l2_train(train_cfg)
    infer_summary = run_l2_infer(infer_cfg)

    all_summaries.append(
        {
            "dataset_id": dataset_id,
            "run_name": run_name,
            "loader_example": {
                "train_pairs": len(train_pairs),
                "x": list(first_batch["x"].shape),
                "y": list(first_batch["y"].shape),
            },
            "train": train_summary,
            "infer": infer_summary,
            "freeze": infer_summary.get("freeze_outputs", {}),
        }
    )

print(json.dumps(all_summaries, ensure_ascii=False, indent=2))